In [ ]:
### Instalar paquetes y librerias

In [ ]:
!pip install supervision ultralytics trackers -q
!pip install tqdm

import supervision as sv
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
from google.colab import drive
import numpy as np
from tqdm import tqdm



drive.mount("/content/drive")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.5 MB/s eta 0:00:00
Mounted at /content/drive


# Obtener recortes de detecciones cada 30 frames

In [ ]:
#FUNCION PARA IMPORTAR LAS DETECCIONES
from pycocotools import mask as mask_utils
import json
import os
import numpy as np

#PARA CARGAR LOS DATOS NUEVAMENTE
def cargar_detecciones_frame(indice_frame:int,ruta_carpeta:str):
    """
    Reconstruye sv.Detections desde el JSON guardado. Igual me la dio Claude
    """
    in_path = os.path.join(ruta_carpeta,f"frame_{indice_frame:08d}.json")
    with open(in_path) as f:
        datos_rle = json.load(f)

    if len(datos_rle) ==0:
        return sv.Detections.empty()
    mascaras,cajas,confianza,clase,tracker_ids = [],[],[],[],[]
    data_acumulado = {}

    for objeto in datos_rle:
        rle = objeto["rle"]
        rle["counts"] = rle["counts"].encode("utf-8") #de vuelta a bytes
        mascaras.append(mask_utils.decode(rle).astype(bool))
        cajas.append(objeto["xyxy"])
        confianza.append(objeto.get("confidence"))
        clase.append(objeto.get("class_id"))
        tracker_ids.append(objeto.get("tracker_id"))

        #DATA
        for clave, valores in objeto.get("data",{}).items():
            data_acumulado.setdefault(clave,[]).append(valores)

    data_final = {clave: np.array(valores) for clave, valores in data_acumulado.items()}

    return sv.Detections(
        xyxy = np.array(cajas,dtype=np.float32),
        mask=np.stack(mascaras),
        confidence = np.array(confianza,dtype=np.float32) if confianza[0] is not None else None,
        class_id=np.array(clase, dtype=int) if clase[0] is not None else None,
        tracker_id=np.array(tracker_ids, dtype=int) if tracker_ids[0] is not None else None,
        data=data_final if data_final else {},
    )

## 1. Importar video y definir ruta de salida, asi como indicar donde se almacenan las segmentaciones

In [ ]:
CARPETA_FRAMES = "/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo"
#CARPETA_OBJETIVO = "/content/drive/MyDrive/PROYECTO SAM 3/datasets/dataset_prueba_imagenes_clustering"
RUTA_VIDEO = "/content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV"

## 2. Guardar las detecciones del primer frame

Se guardaran con nombre en formato como el siguiente

- "#0001_frame0000001"
- "#0002_frame0000001"

In [ ]:
#1. Obtener el primer frame de video
capturador = cv2.VideoCapture(RUTA_VIDEO)
ret,primer_frame = capturador.read()
capturador.release()

#2. Importar las detecciones del primer frame del video
#Se requiere el numero de frame y la ruta de la carpeta

NUMERO_FRAME = 1
deteccion = cargar_detecciones_frame(NUMERO_FRAME,CARPETA_FRAMES)
#print(deteccion)

#3. Guardar las detecciones en la carpeta de prueba

#3.1 Iterar sobre las detecciones para recortar y nombrar

for i,(xyxy,mask,confidence,clase_id,tracker_id,data) in enumerate(deteccion):
    #1. Obtener el nombre, tracker_id, y numero de frame
    print("Tracker ID: ",tracker_id)
    print(xyxy)
    print(i)
    clase = deteccion.data["class_name"][i]
    track_id = tracker_id
    numero_frame = NUMERO_FRAME

    #2. Definir el nombre del archivo y ruta
    nombre = f"{clase}-{track_id:04d}-frame_{NUMERO_FRAME:08d}.png"
    ruta_guardado = os.path.join(CARPETA_OBJETIVO,nombre)

    #3. Extraer el recorte con cv.crop_image
    crop = sv.crop_image(image=primer_frame,xyxy=xyxy)

    #4. Guardar usando openCV y nombre personalizado
    cv2.imwrite(ruta_guardado,crop)
    print(f"Deteccion #{track_id} guardada.")


#Luego el ejemplo anterior lo tengo que re escribir como una funcion


Tracker ID:  0
[285. 478. 412. 627.]
0
Deteccion #0 guardada.
Tracker ID:  1
[ 375. 1216.  497. 1353.]
1
Deteccion #1 guardada.
Tracker ID:  2
[297. 660. 422. 793.]
2
Deteccion #2 guardada.


## 3. Crear una funcion para guardar las imagenes de las detecciones

In [ ]:
def guardar_recortes(imagen,indice_frame,detecciones,carpeta_objetivo):
    numero_detecciones_guardadas = len(detecciones)

    #1. Iterar en las detecciones
    for i,(caja,mascara,confianza,clase_id,tracker_id,data) in enumerate(detecciones):
        nombre_clase = detecciones.data["class_name"][i]
        numero_id = tracker_id
        conf = confianza

        #2. Definir el nombre del archivo = nombreClase_numeroID_indiceFrame
        nombre = f"#{numero_id:04}-frame_{indice_frame:08d}-{nombre_clase}-conf_{conf:04.2f}.png"
        ruta_de_guardado = os.path.join(carpeta_objetivo,nombre)

        #3. Extraer el recorte de la imagen
        recorte = sv.crop_image(image=imagen,xyxy=caja)

        #4. Guardar el archivo con OpenCV
        cv2.imwrite(ruta_de_guardado,recorte)

    return 0




#Esta funcion es especial para guardar solo las IDs que me faltan, si alguna ID
#ya esta dentro de una lista, entonces se la salta
def guardar_recortes_faltantes(imagen,indice_frame,detecciones,carpeta_objetivo):

    lista = [0,1,2,5,7,9,16,17,20,21,23,24,26,28,29,32,33,35,36,38,39,41,43,44,45,
             50,51,54,56,57,58,59,60,61,62,65,66,68,70,71,72,73,74,75,76,77,78,79,
             81,91,92,96,98,112,115,118,119,121,122,123,125,127,155,200,104,205,206,207,
             209,210,212,216,217]

    #1. Iterar en las detecciones
    n=0
    for i,(caja,mascara,confianza,clase_id,tracker_id,data) in enumerate(detecciones):

        if(tracker_id not in lista):
            print("Tracker ID a guardar: ",tracker_id)

            nombre_clase = detecciones.data["class_name"][i]
            numero_id = tracker_id
            conf = confianza

            #2. Definir el nombre del archivo = nombreClase_numeroID_indiceFrame
            nombre = f"#{numero_id:04}-frame_{indice_frame:08d}-{nombre_clase}-conf_{conf:04.2f}.png"
            ruta_de_guardado = os.path.join(carpeta_objetivo,nombre)

            #3. Extraer el recorte de la imagen
            recorte = sv.crop_image(image=imagen,xyxy=caja)

            #4. Guardar el archivo con OpenCV
            cv2.imwrite(ruta_de_guardado,recorte)
            print("Tracker ID guardado: ",tracker_id)
            n = n+1

    return n

In [ ]:
#GUARDAR LAS IMAGENES USANDO LA FUNCION
#guardar_recortes(primer_frame,NUMERO_FRAME,deteccion,CARPETA_OBJETIVO)


0

## 2. Iterar primeros 6 minutos del video y guardar cada 30 frames (cada segundo)

In [ ]:
CARPETA_OBJETIVO_PRUEBA = "/content/drive/MyDrive/PROYECTO SAM 3/datasets/dataset_cluster_ids_faltantes_manuales"
CARPETA_FRAMES = "/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo"
RUTA_VIDEO = "/content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV"


lista_excepciones = [0,1,2,5,7,9,16,17,20,21,23,24,26,28,29,32,33,35,36,38,39,41,43,44,45,
        50,51,54,56,57,58,59,60,61,62,65,66,68,70,71,72,73,74,75,76,77,78,79,
        81,91,92,96,98,112,115,118,119,121,122,123,125,127,155,200,104,205,206,207,
        209,210,212,216,217]

In [ ]:
#INICIALIZAR SOLAMENTE EL CAPTURADOR
capturador = cv2.VideoCapture(RUTA_VIDEO)

#INICIAR CICLO DE LECTURA - ir contando cuantas detecciones se van guardando

#Guardar solo si len(detecciones) > 0

fps_original = capturador.get(cv2.CAP_PROP_FPS)
frames_totales = int(capturador.get(cv2.CAP_PROP_FRAME_COUNT))
stride = 15 #Cada cuanto se van a guardar los frames


frame_inicio = 1  #Numero del frame en el que se desea inicia del video
numero_frames_recorrer = frames_totales+1 #El numero limite de frames a recorrer
capturador.set(cv2.CAP_PROP_POS_FRAMES,frame_inicio)



with tqdm(total=frames_totales,desc="Buscando recortes",unit="frame") as pbar:

    indice_frame = frame_inicio
    total_recortes = 0

    while(capturador.isOpened()):
        ret,frame = capturador.read()
        #print(indice_frame)

        if(not ret or (indice_frame>numero_frames_recorrer)):break

        if (indice_frame%stride==0): #Las detecciones se cargan solo cuando el numero de frame es multiplo del stride

            #Vamos a poner un try catch en caso de que no exista la ruta de archivo
            try:
                detecciones = cargar_detecciones_frame(indice_frame,CARPETA_FRAMES)


                #mascara = detecciones[detecciones.tracker_id not in lista_excepciones]

                #print("Try: detecciones cargadas del frame: ",indice_frame)
                #mascara = detecciones.confidence>0.30 #Filtramos por el valor de confidence
                #detecciones = detecciones[mascara]


                #Cuando hay mascara, el area  es el area de la mascara, sino, es el area del bbox
                #Mejor que sea del bbox
                #detecciones = detecciones[detecciones.area  > 5000]
                #detecciones = detecciones[detecciones.box_area  > 8000] #  Aqui se filtra por el area de la mascara, no del bounding box
                #detecciones = detecciones[detecciones.box_area < 60000]

                n=guardar_recortes_faltantes(frame,indice_frame,detecciones,CARPETA_OBJETIVO_PRUEBA)

                if(n>0):
                    #guardar_recortes(frame,indice_frame,detecciones,CARPETA_OBJETIVO_PRUEBA)
                    total_recortes = total_recortes + n
                    print(f"Fr: {indice_frame}-{n} nuevas dets --- {total_recortes} acumuladas")

            except Exception as error:
                print(f"Error al cargar detecciones fr: {indice_frame}. {error}")
                indice_frame = indice_frame+1
                pbar.update(1)
                continue
        indice_frame = indice_frame+1
        pbar.update(1)

capturador.release()
#no se que mas xd

Buscando recortes:   1%|          | 204/19407 [00:02<05:12, 61.54frame/s]

Tracker ID a guardar:  3
Tracker ID guardado:  3
Fr: 195-1 nuevas dets --- 1 acumuladas


Buscando recortes:   2%|▏         | 372/19407 [00:06<04:22, 72.57frame/s]

Tracker ID a guardar:  4
Tracker ID guardado:  4
Fr: 360-1 nuevas dets --- 2 acumuladas


Buscando recortes:   3%|▎         | 613/19407 [00:09<04:11, 74.63frame/s]

...

Tracker ID a guardar:  314
Tracker ID guardado:  314
Tracker ID a guardar:  318
Tracker ID guardado:  318
Fr: 19350-2 nuevas dets --- 504 acumuladas


Buscando recortes: 100%|█████████▉| 19376/19407 [09:01<00:00, 31.11frame/s]

Tracker ID a guardar:  321
Tracker ID guardado:  321
Tracker ID a guardar:  322
Tracker ID guardado:  322
Fr: 19365-2 nuevas dets --- 506 acumuladas


Buscando recortes: 100%|█████████▉| 19392/19407 [09:02<00:00, 35.76frame/s]

Tracker ID a guardar:  323
Tracker ID guardado:  323
Tracker ID a guardar:  324
Tracker ID guardado:  324
Fr: 19380-2 nuevas dets --- 508 acumuladas


In [ ]:
capturador.release()

# NOTAS Y COSAS A TOMAR EN CUENTA

- Siempre sera buena idea filtrar las detecciones a guardar por area de caja de delimitacion

Cuando en `sv.Detections` hay una mascara, el valor de

`detecciones.area` es el area de pixeles de la segmentacion, si no tiene segmentacion, entonces calcula la de la caja delimitadora.

`detecciones.box_area` si devuelve el valor del area de la caja delimitadora.


## Algunas observaciones sobre el rendimiento de SAM3

Hay que tomar en cuenta que en video la segmentacion la combina con una prediccion de la segmentacion, no analiza el frame desde cero, pues tendria que calcular los embeddings de cada frame.

- a veces detectaba como robot un baston
- la cancha completa
- metia a dos robots en una caja con una sola ID
- detectaba robots cuando ya los quitaban